# 01 · Data Loading, Cleaning & Merge
## Austin Animal Center — Intakes + Outcomes → one analysis-ready table

**Where this fits.** This is the first notebook in the project pipeline:
`01_cleaning` (this file) → `02_eda` → `03_modeling`. It turns the two raw historical
exports into a single cleaned, merged dataset that every later notebook reads from
`../data/processed/df_full_merged.csv`.

**Inputs.** Two local full-history CSV exports (2013-10-01 → 2025-05-05) downloaded from
the Austin Open Data Portal:
- `Austin_Animal_Center_Intakes_…csv` (~173.8k rows) — one row per **intake** event
- `Austin_Animal_Center_Outcomes_…csv` (~173.8k rows) — one row per **outcome** event

**What this notebook does (Steps 1–8).**
1. **Rename & trim columns** — align both exports to a shared `snake_case` schema and drop columns not needed downstream.
2. **Normalize & validate dates** — convert mixed-timezone ISO datetimes to date-only (YYYY-MM-DD) and null out impossible `date_of_birth` values (DOB after `outcome_date`).
3. **Merge** — backward `merge_asof` intakes ← outcomes on `animal_id`, matching each outcome to its most recent prior intake → one row per outcome event.
4. **Clean the merge** — drop orphan outcomes with no matching intake (~0.5%) and deduplicate.
5. **Parse sex & breed** — split `sex_upon_intake` into `sex` + the non-leaky `is_previously_spayed_neutered` (known at intake time), and split `breed` into `primary_breed` + `is_mix`; drop the raw source columns.
6. **Clean color** — split `color` on "/" into `primary_color` / `secondary_color`, then route `primary_color` through dog- vs cat-specific maps to derive a coarse base color and `pattern`.
7. **Engineer baseline features** — `length_of_stay_days`, `age_at_intake_days` (negatives dropped), the **primary target `is_long_stay`** (stay > 30 days, AAC's official long-stay definition), the now comparison-only `is_adopted`, the multi-class `outcome_category`, and intake-date temporal features (`intake_year`, `intake_month`, `intake_weekday`).
8. **Save** — reorder to the canonical column layout, filter to **Dog & Cat**, and write `../data/processed/df_full_merged.csv`.

**Output.** `df_full_merged.csv` — include only **Dog & Cat** (162,765 rows) for analysis and modeling.

**Targets.**
- **Binary (primary) — `is_long_stay`**: 1 = length of stay **> 30 days** (Austin Animal Center's official long-stay definition), else 0. The modeling goal is now to flag animals likely to occupy shelter space long-term so staff can intervene early.
- **`is_adopted`** (1 = Adopted incl. `Rto-Adopt`, else 0) — **no longer a modeling target**; demoted to a comparison-only derived column. Kept in the output, not deleted.

**Leakage caution (set here, enforced in `03_modeling`).** Because the target is now `is_long_stay`, every outcome-derived column is a leakage source and must never be a model feature: `outcome_type`, `outcome_subtype`, `outcome_date`, `length_of_stay_days`, and — newly — `is_adopted` and `outcome_category` (both derived from the outcome). **Note:** `01_cleaning` does **not** drop these; it only records the role change. The actual removal-from-features happens in `03_modeling`. `length_of_stay_days` stays a legitimate derived column here — it is the source of `is_long_stay`.

## Setup


In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

pandas  3.0.0
numpy   2.4.1


## Data Loading

Local copies of the full historical exports (2013-10-01 to 2025-05-05) downloaded
from the Austin data portal UI.

In [35]:
FULL_OUTCOMES_PATH = Path('../data/raw/Austin_Animal_Center_Outcomes_(10_01_2013_to_05_05_2025)_20260523.csv')
FULL_INTAKES_PATH  = Path('../data/raw/Austin_Animal_Center_Intakes_(10_01_2013_to_05_05_2025)_20260523.csv')

df_full_outcomes = pd.read_csv(FULL_OUTCOMES_PATH)
df_full_intakes  = pd.read_csv(FULL_INTAKES_PATH)

print('=== FULL OUTCOMES EXPORT ===')
df_full_outcomes.info()
print('\n--- head(3) ---')
display(df_full_outcomes.head(3))

print('\n=== FULL INTAKES EXPORT ===')
df_full_intakes.info()
print('\n--- head(3) ---')
df_full_intakes.head(3)

=== FULL OUTCOMES EXPORT ===
<class 'pandas.DataFrame'>
RangeIndex: 173775 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Animal ID         173775 non-null  str  
 1   Date of Birth     173775 non-null  str  
 2   Name              123991 non-null  str  
 3   DateTime          173775 non-null  str  
 4   MonthYear         173775 non-null  str  
 5   Outcome Type      173729 non-null  str  
 6   Outcome Subtype   79660 non-null   str  
 7   Animal Type       173775 non-null  str  
 8   Sex upon Outcome  173774 non-null  str  
 9   Age upon Outcome  173766 non-null  str  
 10  Breed             173775 non-null  str  
 11  Color             173775 non-null  str  
dtypes: str(12)
memory usage: 15.9 MB

--- head(3) ---


,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green



=== FULL INTAKES EXPORT ===
<class 'pandas.DataFrame'>
RangeIndex: 173812 entries, 0 to 173811
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Animal ID         173812 non-null  str  
 1   Name              123821 non-null  str  
 2   DateTime          173812 non-null  str  
 3   MonthYear         173812 non-null  str  
 4   Found Location    173812 non-null  str  
 5   Intake Type       173812 non-null  str  
 6   Intake Condition  173812 non-null  str  
 7   Animal Type       173812 non-null  str  
 8   Sex upon Intake   173811 non-null  str  
 9   Age upon Intake   173812 non-null  str  
 10  Breed             173812 non-null  str  
 11  Color             173812 non-null  str  
dtypes: str(12)
memory usage: 15.9 MB

--- head(3) ---


,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
0,A521520,Nina,10/01/2013 07:51:00 AM,October 2013,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,7 years,Border Terrier/Border Collie,White/Tan
1,A664235,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
2,A664236,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White


## Clean & Merge Full-Export CSVs into One Frame

Align the 173k historical CSV rows to a snake_case schema, then backward-asof
merge intakes ← outcomes so each outcome carries its triggering intake's
context. Output: `df_full_merged` — one row per outcome event.

### Step 1: change the column names into sanke_schema for further data cleaning

In [36]:
# --- outcomes side: rename to target schema ---
o = df_full_outcomes.rename(columns={
    'Animal ID':       'animal_id',
    'Name':            'name',
    'DateTime':        'outcome_date',
    'Outcome Type':    'outcome_type',
    'Date of Birth':   'date_of_birth',
    'Outcome Subtype': 'outcome_subtype',
})

o = o.drop(columns=['Sex upon Outcome', 'MonthYear' ,'Age upon Outcome', 'Breed','Animal Type', 'Color']) 

# --- intakes side: rename to target schema ---
i = df_full_intakes.rename(columns={
    'Animal ID':        'animal_id',
    'DateTime':         'intake_date',
    'Found Location':   'found_location',
    'Intake Type':      'intake_reason',
    'Intake Condition': 'intake_health_condition',
    'Breed':            'breed',
    'Animal Type':      'animal_type',
    'Color':            'color',       
    'Sex upon Intake':  'sex_upon_intake', 
})

# 'Name' kept from the outcomes side only, so drop it here to avoid a duplicate column
i = i.drop(columns=['MonthYear', 'Age upon Intake', 'Name'])

print(f'After rename — outcomes: {o.shape}, intakes: {i.shape}')
print('Outcomes cols:', list(o.columns))
print('Intakes  cols:', list(i.columns))

After rename — outcomes: (173775, 6), intakes: (173812, 9)
Outcomes cols: ['animal_id', 'date_of_birth', 'name', 'outcome_date', 'outcome_type', 'outcome_subtype']
Intakes  cols: ['animal_id', 'intake_date', 'found_location', 'intake_reason', 'intake_health_condition', 'animal_type', 'sex_upon_intake', 'breed', 'color']


### Step 2: Normalize & Validate dates
Convert mixed ISO datetimes to date-only (YYYY-MM-DD), stripping timezone offsets. Set any date_of_birth that is after outcome_date into null.

In [ ]:
# Parse mixed ISO datetimes into date-only (YYYY-MM-DD)
# Reasons:
# - Source strings contain both timezone-aware (e.g. 2013-12-02T00:00:00-05:00) and naive ISO strings (e.g. 2013-10-01T09:31:00).
# - If pandas parses a Series as tz-aware, the naive strings can become NaT.
# - To avoid that, strip the timezone offset suffix, parse, then normalize to date.

def _parse_date_only(series: pd.Series) -> pd.Series:
    """Return a date-only Series (datetime64[ns]) from mixed ISO datetime strings."""
    series = series.astype(str).str.strip()
    # Remove timezone offsets like -05:00 or +00:00 so both offset and naive strings parse
    series = series.str.replace(r'([+-]\d{2}):(\d{2})$', '', regex=True)
    # Parse; errors='coerce' turns unparseable values into NaT, .dt.normalize() keeps date only
    return pd.to_datetime(series, errors='coerce').dt.normalize()

# Apply parsing to columns (works whether the column currently holds raw strings
# or Timestamp objects represented as strings)
o['outcome_date'] = _parse_date_only(o['outcome_date'])
o['date_of_birth'] = _parse_date_only(o['date_of_birth'])
i['intake_date'] = _parse_date_only(i['intake_date'])

# DOB after outcome is impossible — null these (compare dates only)
mask = o['date_of_birth'] > o['outcome_date']
o.loc[mask, 'date_of_birth'] = pd.NaT

# Audits
print(f'Nulled impossible DOBs: {o["date_of_birth"].isna().sum():,}')
print(f'Null outcome dates: {o["outcome_date"].isna().sum():,}')
print(f'Null intake dates: {i["intake_date"].isna().sum():,}')

Nulled impossible DOBs: 33
Null outcome dates: 0
Null intake dates: 0


### Step 3: Merge intakes ← outcomes & engineer baseline features and srop no intake records

Backward `merge_asof` on `animal_id`, matching each outcome to the most recent
prior intake (`direction='backward'`), so every outcome row carries its
triggering intake's context. Output: `df_full_merged` — one row per outcome event.

In [38]:
intake_cols = [
    'animal_id', 
    'intake_date', 
    'found_location', 
    'intake_reason', 
    'intake_health_condition', 
    'animal_type', 
    'sex_upon_intake', 
    'breed', 
    'color']

intakes_for_merge = (
    i[intake_cols]
    .sort_values('intake_date')
)
outcomes_sorted = o.sort_values('outcome_date')

df_full_merged = pd.merge_asof(
    outcomes_sorted,
    intakes_for_merge,
    by='animal_id',
    left_on='outcome_date',
    right_on='intake_date',
    direction='backward',
)


### Step 4: Drop unmatched outcomes and deduplicate merged data

Filter out merged outcome rows that have no matched prior `intake_date`, then remove duplicate rows from `df_full_merged`. Print the number of rows dropped and audit the final row count and missing intake dates.

In [39]:
# drop orphans with missing intake date (~0.5%)
before = len(df_full_merged)
df_full_merged = df_full_merged[df_full_merged['intake_date'].notna()].copy()
dropped = before - len(df_full_merged)
print(f'Dropped {dropped:,} rows missing intake_date '
      f'({dropped / before * 100:.2f}%) — {len(df_full_merged):,} rows remain')

# drop duplicates
print(f'Drop duplicates:             {df_full_merged.duplicated().sum():,}')
df_full_merged.drop_duplicates(inplace=True)

# Audits
print(f'Row parity (merged vs outcomes): {len(df_full_merged):,} vs {len(o):,}')
print(f'Missing intake_date:             {df_full_merged["intake_date"].isna().sum():,}')

Dropped 925 rows missing intake_date (0.53%) — 172,850 rows remain
Drop duplicates:             47
Row parity (merged vs outcomes): 172,803 vs 173,775
Missing intake_date:             0


### Step 5: Parse sex, spay/neuter and breed fields

Extract `sex` from `sex_upon_outcome` — but **drop the outcome spay/neuter flag**: it's
target leakage (shelters spay/neuter as part of adoption, so the flag is caused by the
outcome). Split `breed` into primary/secondary breeds, derive the
non-leaky boolean `is_previously_spayed_neutered` from `sex_upon_intake` (known at intake
time), and drop the raw source columns.

In [40]:
SEX_UPON_MAP = {
    'Neutered Male': ('Male',    'Yes'),
    'Spayed Female': ('Female',  'Yes'),
    'Intact Male':   ('Male',    'No'),
    'Intact Female': ('Female',  'No'),
    'Unknown':       ('Unknown', 'Unknown'),
}


def _split_sex(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    mapped = series.map(SEX_UPON_MAP)
    sex  = mapped.map(lambda t: t[0] if isinstance(t, tuple) else 'Unknown')
    spay = mapped.map(lambda t: t[1] if isinstance(t, tuple) else 'Unknown')
    return sex, spay

# split breed into primary breed and is_mix (1/0)
def parse_breed(b):
    if pd.isna(b):
        return (np.nan, pd.NA)
    b = str(b).strip()
    # 1 = mixed/crossbreed (has " Mix" or a "/"), 0 = single breed
    is_mix = 1 if (('mix' in b.lower()) or ('/' in b)) else 0
    # Take first breed token (before "/" or " Mix")
    primary = b.split('/')[0].replace(' Mix', '').replace(' mix', '').strip()
    return (primary, is_mix)

# --- split Sex upon Intake -> sex + is_previously_spayed_neutered ---
df_full_merged['sex'], _ = _split_sex(df_full_merged['sex_upon_intake'])
_, spay_intake = _split_sex(df_full_merged['sex_upon_intake'])
df_full_merged['is_previously_spayed_neutered'] = spay_intake.map({'Yes': 1, 'No': 0}).astype('Int64')
df_full_merged = df_full_merged.drop(columns=['sex_upon_intake'])


# --- breed -> primary breed + is_mix (1/0) ---
parsed = df_full_merged['breed'].apply(parse_breed)
df_full_merged['primary_breed'] = [p[0] for p in parsed]
df_full_merged['is_mix']        = pd.array([p[1] for p in parsed], dtype='Int64')
df_full_merged = df_full_merged.drop(columns=['breed'])


### Step 6: Clean `color` column

Split `color` into `primary_color`, `primary_pattern` and `secondary_color`. The split logic will be specified for different color ditribution of dogs and cats.

In [41]:
TOPN = 10
COLOR_COL = 'color'   # raw/original color, before to_color_group

# --- tables: top-N raw colors per species ---
for sp in ['Dog', 'Cat']:
    vc = df_full_merged.loc[df_full_merged['animal_type'] == sp, COLOR_COL].value_counts()
    print(f'\n=== {sp}: {vc.size} distinct {COLOR_COL} (showing top {TOPN}) ===')
    print(vc.head(TOPN).to_string())



=== Dog: 400 distinct color (showing top 10) ===
color
Black/White    11140
Brown/White     5347
White           5170
Tan/White       4914
Black           4910
Tan             4081
Brown           3826
Black/Tan       3607
Tricolor        3445
Black/Brown     3414

=== Cat: 327 distinct color (showing top 10) ===
color
Brown Tabby           10641
Black                  8934
Black/White            6198
Brown Tabby/White      5424
Orange Tabby           5114
Tortie                 3153
Calico                 2960
Blue Tabby             2794
Orange Tabby/White     2554
Blue                   2547


In [42]:
def _split_slash(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    parts = series.fillna('').str.split('/', n=1, expand=True)
    primary = parts[0].replace('', np.nan)
# If no value contains '/', split yields a single column. Return an all-NaN
# Series (not a bare scalar) so the secondary_color assignment stays aligned.

    if parts.shape[1] > 1:
        secondary = parts[1].replace('', np.nan)
    else:
        secondary = pd.Series(np.nan, index=series.index)
    return primary, secondary

df_full_merged['primary_color'], df_full_merged['secondary_color'] = _split_slash(df_full_merged['color'])
df_full_merged = df_full_merged.drop(columns=['color'])

In [43]:
# ===== Dog route =====
DOG_COLOR_BASE = {
    'Black':'Black', 'White':'White',
    'Brown':'Brown', 'Chocolate':'Brown', 'Sable':'Brown', 'Liver':'Brown',
    'Tan':'Tan', 'Fawn':'Tan', 'Buff':'Tan', 'Apricot':'Tan', 'Gold':'Tan', 'Yellow':'Tan',
    'Blue':'Blue/Gray', 'Gray':'Blue/Gray', 'Silver':'Blue/Gray',
    'Red':'Red/Orange', 'Orange':'Red/Orange',
    'Cream':'Cream',
    'Tricolor':'Tricolor', 'Tricolored':'Tricolor',
}
DOG_PATTERN_KEYS      = ['Brindle', 'Merle', 'Tick', 'Tiger']
DOG_PATTERN_NORMALIZE = {'Tiger': 'Brindle'}

# ===== Cat route =====
CAT_COLOR_BASE = {
    'Black':'Black', 'White':'White',
    'Blue':'Blue/Gray', 'Gray':'Blue/Gray', 'Silver':'Blue/Gray', 'Lilac':'Blue/Gray',
    'Cream':'Cream', 'Orange':'Orange',
    'Seal':'Point-tint', 'Flame':'Point-tint', 'Lynx':'Point-tint',
    'Brown':'Brown', 'Chocolate':'Brown',                          # plain choc -> Brown; Choc Point -> base Brown + pattern Point
    'Torbie':'Tortie', 'Tortie':'Tortie',         # Tortie family treated as base (see markdown)
    'Calico':'Calico',
}
CAT_PATTERN_KEYS      = ['Tabby', 'Point', 'Tiger', 'Smoke']
CAT_PATTERN_NORMALIZE = {'Tiger': 'Tabby'}

def split_base_pattern(c, color_base, pattern_keys, pattern_normalize):
    """Return (base_color, pattern) using the species-specific maps.
       NaN->('Unknown','Unknown'); no pattern word -> 'Solid';
       pattern word but no base word -> base='None'."""
    if pd.isna(c):
        return ('Unknown', 'Unknown')
    c = str(c).strip()

    pattern = next((p for p in pattern_keys if p in c), 'Solid')
    pattern = pattern_normalize.get(pattern, pattern)

    base_str = c
    for p in pattern_keys:
        base_str = base_str.replace(p, '')
    # also strip any base-words that are really pattern-family labels (Tortie/Calico/Torbie)
    # so the remaining token is the true tint; keep them only if nothing else remains.
    base_str = base_str.strip()
    base = color_base.get(base_str.split()[0], 'Other') if base_str else 'None'
    return (base, pattern)

def route_split(row):
    if row['animal_type'] == 'Dog':
        return split_base_pattern(row['primary_color'], DOG_COLOR_BASE, DOG_PATTERN_KEYS, DOG_PATTERN_NORMALIZE)
    elif row['animal_type'] == 'Cat':
        return split_base_pattern(row['primary_color'], CAT_COLOR_BASE, CAT_PATTERN_KEYS, CAT_PATTERN_NORMALIZE)
    return ('Unknown', 'Unknown')

_parsed = df_full_merged.apply(route_split, axis=1)
df_full_merged['primary_color'] = _parsed.map(lambda t: t[0])
df_full_merged['pattern']    = _parsed.map(lambda t: t[1])

print('Added base_color / pattern via dog & cat routes.')

Added base_color / pattern via dog & cat routes.


### Step 7: Compute baseline features
Add baseline features
- **`length_of_stay_days`** — `outcome_date − intake_date`
- **`is_long_stay`** — **primary target**, 1 if `length_of_stay_days > 30`, else 0
- **`age_at_intake_days`** — `intake_date − date_of_birth`
- **`is_adopted`** — comparison-only, 1 if adopted else 0
- **`outcome_category`** - combine into 'adopted', 'return to owner', 'transfer', 'death', 'other'
- **`intake_year`** - year of intake date
- **`intake_month`** - month of intake date
- **`intake_weekday`** - intake day of week

In [ ]:
# Derived column- 'length_of_stay_days'
df_full_merged['length_of_stay_days'] = (
    df_full_merged['outcome_date'] - df_full_merged['intake_date']
).dt.days
print(f'Negative length of stay: {(df_full_merged["length_of_stay_days"] < 0).sum():,} days')
print(f'Median length of stay:   {df_full_merged["length_of_stay_days"].median():.0f} days')
print('='*60)

# Derived column- 'is_long_stay' (PRIMARY TARGET): AAC official long-stay = stay over 30 days (strict >)
LONG_STAY_THRESHOLD = 30
df_full_merged['is_long_stay'] = (
    df_full_merged['length_of_stay_days'] > LONG_STAY_THRESHOLD
).astype(int)
print('is_long_stay rate by animal_type:')
print(df_full_merged.groupby('animal_type')['is_long_stay'].mean())
print('='*60)

# Derived column- 'age_at_intake_days'
df_full_merged['age_at_intake_days'] = (
    df_full_merged['intake_date'] - df_full_merged['date_of_birth']
).dt.days
before = len(df_full_merged)
df_full_merged = df_full_merged[~(df_full_merged['age_at_intake_days'] < 0)].copy()
dropped = before - len(df_full_merged)
df_full_merged['age_at_intake_days'] = df_full_merged['age_at_intake_days'].astype('Int64')
print(f'Dropped {dropped:,} rows with negative age_at_intake_days '
      f'({dropped / before * 100:.2f}%) — {len(df_full_merged):,} rows remain')
print('='*60)


# Derived column- 'is_adopted'
# Group the 12 raw outcome_type values into 5 modeling categories.
OUTCOME_CATEGORY_MAP = {
    'Adoption':        'adopted',
    'Rto-Adopt':       'adopted',
    'Return to Owner': 'return to owner',
    'Transfer':        'transfer',
    'Euthanasia':      'died',
    'Disposal':        'died',
    'Died':            'died',
}

# 'is_adopted' is the target variable for modeling (1 if adopted or Rto-Adopt, else 0)
df_full_merged['is_adopted'] = df_full_merged['outcome_type'].isin(['Adoption', 'Rto-Adopt']).astype(int)
df_full_merged['outcome_category'] = df_full_merged['outcome_type'].map(OUTCOME_CATEGORY_MAP).fillna('other')

# Derived temporal columns
df_full_merged['intake_year'] = df_full_merged['intake_date'].dt.year
df_full_merged['intake_month'] = df_full_merged['intake_date'].dt.month
df_full_merged['intake_weekday'] = df_full_merged['intake_date'].dt.weekday

Negative length of stay: 0 days
Median length of stay:   6 days
is_long_stay rate by animal_type:
animal_type
Bird        0.07
Cat         0.25
Dog         0.16
Livestock   0.45
Other       0.04
Name: is_long_stay, dtype: float64
Dropped 191 rows with negative age_at_intake_days (0.11%) — 172,612 rows remain


### Step 7b: Data-quality spot-check — very long stays (`length_of_stay_days > 365`)


In [49]:
LONG_STAY_AUDIT_DAYS = 365
longstay = df_full_merged[df_full_merged['length_of_stay_days'] > LONG_STAY_AUDIT_DAYS].copy()
n = len(longstay)
print(f'length_of_stay_days > {LONG_STAY_AUDIT_DAYS}: {n:,} rows '
      f'({n / len(df_full_merged) * 100:.2f}% of merged)')

# Times each animal_id appears in the RAW exports
intake_visits  = df_full_intakes['Animal ID'].value_counts()
outcome_visits = df_full_outcomes['Animal ID'].value_counts()
longstay['n_intakes']  = longstay['animal_id'].map(intake_visits).fillna(0).astype(int)
longstay['n_outcomes'] = longstay['animal_id'].map(outcome_visits).fillna(0).astype(int)

multi = (longstay['n_intakes'] > 1) | (longstay['n_outcomes'] > 1)
print(f'  multi-visit (>1 intake or >1 outcome) —  {multi.sum():,} '
      f'({multi.mean()*100:.1f}% of the >365d rows)')
print(f'  single-visit long stays:                   {(~multi).sum():,}')

cols = ['animal_id', 'animal_type', 'intake_date', 'outcome_date',
        'length_of_stay_days', 'n_intakes', 'n_outcomes', 'outcome_type']
print('\nTop 15 longest stays (for manual review):')
display(longstay.sort_values('length_of_stay_days', ascending=False)[cols].head(15))

length_of_stay_days > 365: 568 rows (0.35% of merged)
  multi-visit (>1 intake or >1 outcome) —  181 (31.9% of the >365d rows)
  single-visit long stays:                   387

Top 15 longest stays (for manual review):


,animal_id,animal_type,intake_date,outcome_date,length_of_stay_days,n_intakes,n_outcomes,outcome_type
125143,A642712,Dog,2016-01-05,2021-04-01,1913,1,1,Adoption
144862,A764237,Cat,2017-12-26,2022-10-06,1745,1,1,Adoption
117761,A722987,Dog,2016-03-24,2020-05-24,1522,1,1,Adoption
100274,A702369,Cat,2015-05-12,2019-05-21,1470,1,1,Adoption
100662,A707965,Cat,2015-07-20,2019-05-28,1408,1,1,Adoption
156101,A810081,Dog,2020-01-21,2023-10-10,1358,2,2,Adoption
118897,A737705,Dog,2016-11-02,2020-07-16,1352,1,1,Transfer
125190,A762152,Cat,2017-11-15,2021-04-03,1235,2,2,Adoption
134063,A778611,Dog,2018-08-17,2021-11-24,1195,1,1,Adoption
170023,A843014,Dog,2021-09-25,2024-12-27,1189,1,1,Adoption


### Step 8: Store processed data into csv

In [46]:
# Reorder columns to the requested canonical order
desired_order = [
    'animal_id',
    'name',
    'animal_type',
    'sex',
    'primary_breed',
    'is_mix',
    'date_of_birth',
    'primary_color',
    'pattern',
    'secondary_color',
    'intake_date',
    'intake_reason',
    'found_location',
    'age_at_intake_days',
    'intake_health_condition',
    'is_previously_spayed_neutered',
    'outcome_date',
    'outcome_type',
    'outcome_subtype',
    'length_of_stay_days',
    'is_long_stay',
    'is_adopted',
    'outcome_category',
    'intake_year',
    'intake_month',
    'intake_weekday',
]

existing = [c for c in desired_order if c in df_full_merged.columns]
missing = [c for c in desired_order if c not in df_full_merged.columns]
if missing:
    print('Warning: missing columns not found and will be skipped:', missing)

# Preserve any other columns that are not in the requested list
other_cols = [c for c in df_full_merged.columns if c not in existing]

# Reassign with new column order
df_full_merged = df_full_merged[existing + other_cols]
print('Reordered df_full_merged columns — first columns now:')
print(df_full_merged.columns.tolist()[:len(existing)])

#check unique values of animal_type and filter to contain only dogs and cats for EDA
df_full_merged['animal_type'].unique() 
df_full_merged= df_full_merged[df_full_merged['animal_type'].isin(['Dog', 'Cat'])]

processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

out_path = processed_dir / 'df_full_merged.csv'
df_full_merged.to_csv(out_path, index=False)

print(f'Saved {len(df_full_merged):,} rows to {out_path}')

Reordered df_full_merged columns — first columns now:
['animal_id', 'name', 'animal_type', 'sex', 'primary_breed', 'is_mix', 'date_of_birth', 'primary_color', 'pattern', 'secondary_color', 'intake_date', 'intake_reason', 'found_location', 'age_at_intake_days', 'intake_health_condition', 'is_previously_spayed_neutered', 'outcome_date', 'outcome_type', 'outcome_subtype', 'length_of_stay_days', 'is_long_stay', 'is_adopted', 'outcome_category', 'intake_year', 'intake_month', 'intake_weekday']
Saved 162,765 rows to ../data/processed/df_full_merged.csv


In [47]:
df_full_merged.head()
df_full_merged.info()

<class 'pandas.DataFrame'>
Index: 162765 entries, 1 to 173774
Data columns (total 26 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   animal_id                      162765 non-null  str           
 1   name                           121169 non-null  str           
 2   animal_type                    162765 non-null  str           
 3   sex                            162765 non-null  str           
 4   primary_breed                  162765 non-null  str           
 5   is_mix                         162765 non-null  Int64         
 6   date_of_birth                  162734 non-null  datetime64[us]
 7   primary_color                  162765 non-null  str           
 8   pattern                        162765 non-null  str           
 9   secondary_color                87231 non-null   str           
 10  intake_date                    162765 non-null  datetime64[us]
 11  intake_reason   